# 01 · Data pipeline: building the training set

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baz08/crypto-sentiment-bert/blob/main/notebooks/01_data_pipeline.ipynb)

Merges four labeled sources of r/CryptoCurrency comments and posts into one training file,
`Crypto_c.csv`, with the columns the model trains on:

| column | meaning |
|---|---|
| `DATA_COLUMN` | cleaned text |
| `LABEL_COLUMN` | 0 = negative, 1 = neutral, 2 = positive |
| `source` | which input file the row came from |

Runs on a CPU runtime in under a minute. `Crypto_c.csv` is already checked into the repo, so you only need
this notebook if you change the cleaning or the sources. Notebook `02_train_evaluate` picks up from here.

In [ ]:
# Get the repo (for the labeled data). Change BRANCH to test an unmerged branch.
REPO_URL = "https://github.com/baz08/crypto-sentiment-bert"
BRANCH = "main"

import os
if not os.path.exists("crypto-sentiment-bert"):
    !git clone -q --depth 1 -b {BRANCH} {REPO_URL}
REPO_DIR = "crypto-sentiment-bert"
DATA_DIR = f"{REPO_DIR}/deployment/reddit"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

LABELS = ["Negative", "Neutral", "Positive"]

## 1. Cleaning

BERT was pretrained on natural text, so stopwords, punctuation and casing all carry signal ("not good" vs. "good",
"!!!"). The cleaning therefore only removes pure noise:

- URLs
- HTML entities left over from Reddit's markdown (`&gt;`, `&amp;#x200B;`)
- **mojibake**: UTF-8 text that was mis-decoded somewhere along the way (`isnâ€™t` → `isn’t`)
- Markdown headers and extra whitespace

This is the exact code in `deployment/reddit/textprep.py`. The API runs the same function on incoming requests,
so the model sees the same kind of text in production that it was trained on.

In [ ]:
"""Light text cleaning applied to every comment before it reaches BERT.

This is the cleaning used for BOTH training (``merge_clean.py``) and
inference (``deployment/api/ML/textprep.py`` is an identical copy, kept in
sync by ``tests/test_textprep.py``). Using the same function in both places
avoids train/serve skew: the model sees the same kind of text in production
that it was trained on.

It is deliberately much lighter than ``cleaner.normalize_corpus``. BERT was
pretrained on natural text, so stopwords, punctuation, and casing all carry
signal ("not good" vs "good", "!!!"), and ``bert-base-uncased`` lowercases
on its own. We only remove things that are pure noise: URLs, HTML entities
left over from Reddit's markdown, encoding artifacts, and extra whitespace.

Standard library only, so it can be imported by the API container and the
data pipeline without extra dependencies.
"""

import html
import re

URL_PATTERN = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
MARKDOWN_HEADER_PATTERN = re.compile(r"^#{1,6}\s*", flags=re.MULTILINE)
WHITESPACE_PATTERN = re.compile(r"\s+")

# Characters that show up when UTF-8 text was decoded as cp1252/latin-1
# (e.g. "isnâ€™t" instead of "isn't").
MOJIBAKE_MARKERS = ("â€", "Ã", "Â")


def fix_mojibake(text):
    """Best-effort repair of UTF-8 text that was mis-decoded as cp1252."""
    if not any(marker in text for marker in MOJIBAKE_MARKERS):
        return text
    try:
        # cp1252 leaves five byte values undefined (0x81, 0x8D, 0x8F, 0x90,
        # 0x9D); those survive mis-decoding as raw control characters, so
        # fall back to latin-1 for them byte-by-byte.
        raw = b"".join(_to_byte(ch) for ch in text)
        return raw.decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text  # not actually mojibake (or mixed); leave it alone


def _to_byte(ch):
    try:
        return ch.encode("cp1252")
    except UnicodeEncodeError:
        return ch.encode("latin-1")


def clean_for_model(text):
    """Normalize a single comment for BERT. Returns '' for missing input."""
    if not isinstance(text, str):
        return ""
    text = fix_mojibake(text)
    # Reddit exports are sometimes double-escaped ("&amp;#x200B;"), so unescape twice.
    text = html.unescape(html.unescape(text))
    text = text.replace("​", " ")
    text = URL_PATTERN.sub(" ", text)
    text = MARKDOWN_HEADER_PATTERN.sub("", text)
    text = WHITESPACE_PATTERN.sub(" ", text)
    return text.strip()

In [ ]:
examples = [
    "&gt; Bitcoin is digital gold https://example.com/article",
    "Decentralization isnâ€™t the future â€” it is now",
    "This is NOT good!!",
]
for text in examples:
    print(f"{text!r}\n  -> {clean_for_model(text)!r}\n")

## 2. Load the four sources

| file | what it is | text column | label column |
|---|---|---|---|
| `BTC_comments.csv` | r/CryptoCurrency comments mentioning BTC (2022) | `body` | `Sentiment` |
| `ETH_comments.csv` | r/CryptoCurrency comments mentioning ETH (2022) | `body` | `Sentiment` |
| `sentiment.csv` | labeled comments, positive/negative only | `Comment Text` | `Sentiment` |
| `sentiment_labels.csv` | labeled posts | `text` | `sentiment` |

The BTC/ETH comments were collected with the Pushshift API (`redditpushshift.py`). Reddit shut off public
Pushshift access in 2023, so these CSVs are a frozen snapshot.

In [ ]:
SOURCES = {
    "BTC_comments.csv": ("body", "Sentiment"),
    "ETH_comments.csv": ("body", "Sentiment"),
    "sentiment.csv": ("Comment Text", "Sentiment"),
    "sentiment_labels.csv": ("text", "sentiment"),
}

def load_source(path, text_column, label_column):
    df = pd.read_csv(path)
    return pd.DataFrame({
        "DATA_COLUMN": df[text_column],
        "LABEL_COLUMN": df[label_column],
        "source": os.path.splitext(os.path.basename(path))[0],
    })

frames = [load_source(f"{DATA_DIR}/{name}", *cols) for name, cols in SOURCES.items()]
raw = pd.concat(frames, ignore_index=True)
print(f"{len(raw)} raw rows")
raw.groupby("source")["LABEL_COLUMN"].apply(lambda s: s.isna().sum()).rename("missing labels").to_frame()

## 3. Merge and clean

Drop rows with no label (or a label outside 0/1/2), rows that are empty after cleaning (for example a comment
that was only a link), and duplicate texts that appear in more than one source.

In [ ]:
def build_training_frame(frames):
    df = pd.concat(frames, ignore_index=True)

    df = df.dropna(subset=["LABEL_COLUMN"])
    df["LABEL_COLUMN"] = df["LABEL_COLUMN"].astype(int)
    df = df[df["LABEL_COLUMN"].isin({0, 1, 2})]

    df["DATA_COLUMN"] = df["DATA_COLUMN"].map(clean_for_model)
    df = df[df["DATA_COLUMN"].str.len() > 0]

    df = df.drop_duplicates(subset="DATA_COLUMN", keep="first")
    return df.reset_index(drop=True)

data = build_training_frame(frames)
print(f"{len(data)} rows after cleaning ({len(raw) - len(data)} dropped)")

counts = pd.crosstab(data["source"], data["LABEL_COLUMN"].map(dict(enumerate(LABELS))), margins=True, margins_name="total")
counts

## 4. Look at the data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Label mix per source
share = pd.crosstab(data["source"], data["LABEL_COLUMN"].map(dict(enumerate(LABELS))), normalize="index")
share[LABELS].plot.barh(stacked=True, ax=axes[0], color=["#c0504d", "#a5a5a5", "#4f81bd"], width=0.7)
axes[0].set_xlabel("share of rows")
axes[0].set_ylabel("")
axes[0].set_title("Label mix by source")
axes[0].legend(loc="lower right", frameon=False)

# Length in words (BERT sees at most 256 tokens; roughly 190 words)
words = data["DATA_COLUMN"].str.split().str.len()
axes[1].hist(words.clip(upper=400), bins=40, color="#4f81bd")
axes[1].axvline(190, color="black", linestyle="--", linewidth=1)
axes[1].text(195, axes[1].get_ylim()[1] * 0.9, "≈ 256-token limit", fontsize=9)
axes[1].set_xlabel("words per row (clipped at 400)")
axes[1].set_title("Text length")

plt.tight_layout()
plt.show()

print(f"median length: {words.median():.0f} words; {(words > 190).mean():.1%} of rows are likely truncated")

**What to notice**

- **The sources disagree on what "neutral" means.** `sentiment.csv` has no neutral class at all, while most of
  `sentiment_labels.csv` is neutral. Mixing labeling schemes adds noise, so check per-source accuracy after training.
- **Neutral is the largest class** (~47%), so a model that always says "neutral" already gets ~47% accuracy.
  That is the bar BERT has to clear, which is why notebook 02 reports macro-F1 next to accuracy.
- Most rows are short comments, but long posts get truncated at 256 tokens.

## 5. Save

In [ ]:
data.to_csv("Crypto_c.csv", index=False)
print(f"saved {len(data)} rows to Crypto_c.csv")

# On Colab: download the file (or copy it into deployment/reddit/ in the repo and commit it).
try:
    from google.colab import files
    files.download("Crypto_c.csv")
except ImportError:
    pass